# py-SCORPIUS — function-by-function R⇄Python parity dictionary

For users migrating existing SCORPIUS R code to Python. Each public R function appears with parameter table + side-by-side calls on the same input + numerical comparison.

Follows the [Notebook 3 schema](https://github.com/omicverse/omicverse-rebuildr/blob/main/NOTEBOOKS.md#notebook-3--function_by_function_r_parityipynb).

## 1. Setup

In [1]:
import os, subprocess, json, sys
for k in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'): os.environ[k]='8'
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

NB = Path('.').resolve()
PORT = NB.parent if NB.name == 'examples' else NB
sys.path.insert(0, str(PORT))
sys.path.insert(0, str(PORT.parent / 'omicverse-rebuildr' / 'engine'))
from parity_metrics import compute_parity
import pyscorpius

R_DUMP = PORT / 'examples' / '_r_outputs'
if not R_DUMP.exists() or not list(R_DUMP.glob('*.json')):
    R_ENV = os.environ.get('R_TEST_ENV', '/scratch/users/steorra/env/CMAP')
    subprocess.run(['conda','run','-p',R_ENV,'Rscript','examples/r_per_function_dump.R'],
                   check=True, cwd=PORT)
def load_r(n): return json.loads((R_DUMP/n).read_text())

# Reload the same simulated dataset (R-generated; saved as CSV sidecar)
expression = pd.read_csv(PORT/'data'/'fixture_simdata_expression.csv', index_col=0).to_numpy(dtype=np.float64)
print(f"expression shape: {expression.shape}")
print(f"R outputs: {sorted(p.name for p in R_DUMP.glob('*.json'))}")

expression shape: (400, 200)
R outputs: ['gene_importances.json', 'infer_trajectory.json', 'reduce_dimensionality.json']


## 2. Function-by-function

<a id='2.1'></a>
### 2.1 `reduce_dimensionality` — pairwise distance + MDS

> Compute pairwise distances between samples, embed via classical / landmark MDS.

**Parameters**:

| R name | Python name | Type | Default | Range | Description |
|---|---|---|---|---|---|
| `x` | `x` | matrix / DataFrame | — | n × p | input expression matrix; rows = samples, cols = features |
| `dist` | `dist` | string | `"spearman"` | one of `"spearman", "pearson", "euclidean", "cosine", "manhattan"` | distance metric |
| `ndim` | `ndim` | int | `3` | ≥ 1 | output dimensionality |
| `num_landmarks` | `num_landmarks` | int | `1000` | ≥ 1 | landmark-MDS threshold (R uses `lmds`; Py uses classical MDS for n ≤ num_landmarks, otherwise simple landmark scheme) |

**R one-liner**:

```r
space <- SCORPIUS::reduce_dimensionality(expression, dist = "spearman", ndim = 2, num_landmarks = 1000)
```

**Python equivalent**:

In [2]:
space = pyscorpius.reduce_dimensionality(expression, dist='spearman', ndim=2, num_landmarks=1000)
print(f"space shape: {space.shape}")

space shape: (400, 2)


**Numerical comparison vs R**:

In [3]:
r_rd = load_r('reduce_dimensionality.json')
r_space = np.array(r_rd['space'])
proc = compute_parity(r_space, space, 'embedding')
print(f"Procrustes similarity: {proc:.4f}  →  {'✅ exact' if proc > 0.95 else '🟡 ' + str(proc)}")

Procrustes similarity: 0.9994  →  ✅ exact


<a id='2.2'></a>
### 2.2 `infer_trajectory` — k-means + TSP + principal curve

> Find a smooth path through the low-dim space; assign each cell an arc-length pseudotime.

**Parameters**:

| R name | Python name | Type | Default | Range | Description |
|---|---|---|---|---|---|
| `space` | `space` | matrix | — | n × ndim | from `reduce_dimensionality` |
| `k` | `k` | int | `4` | ≥ 2 | k-means cluster count for initial trajectory |
| `(princurve params)` | `max_iter`, `smooth_frac` | int, float | 10, 0.5 | — | principal-curve refinement controls |
| — | `seed` | int | `42` | — | **new in Python** — RNG for k-means + TSP |

**R one-liner**:

```r
traj <- SCORPIUS::infer_trajectory(space, k = 4)
```

**Python equivalent**:

In [4]:
traj = pyscorpius.infer_trajectory(space, k=4, seed=42)
print(f"pseudotime: shape={traj['time'].shape} range=[{traj['time'].min():.3f}, {traj['time'].max():.3f}]")
print(f"path: shape={traj['path'].shape}")

pseudotime: shape=(400,) range=[0.000, 1.000]
path: shape=(100, 2)


**Numerical comparison vs R**:

In [5]:
r_it = load_r('infer_trajectory.json')
r_pt = np.array(r_it['time']); p_pt = traj['time']
from scipy.stats import pearsonr
fwd = pearsonr(r_pt, p_pt)[0]; rev = pearsonr(r_pt, 1 - p_pt)[0]
best = max(abs(fwd), abs(rev))
print(f"pseudotime Pearson (best of fwd/rev): {best:.4f}  →  {'✅ ' if best > 0.95 else '🟡 '}{best:.4f}")

pseudotime Pearson (best of fwd/rev): 0.9891  →  ✅ 0.9891


<a id='2.3'></a>
### 2.3 `gene_importances` — random-forest feature importance

> Rank genes by importance for predicting pseudotime.

**Parameters**:

| R name | Python name | Type | Default | Range | Description |
|---|---|---|---|---|---|
| `expression` | `expression` | matrix | — | genes × cells (R) | input expression |
| `time` | `pseudotime` | vector | — | n_cells | pseudotime per cell |
| `num_permutations` | — | int | `0` | — | permutation-importance count (not exposed in Py; uses Gini) |
| — | `n_trees` | int | `500` | ≥ 1 | **new in Python** — RF tree count (R `ranger` default) |
| — | `seed` | int | `42` | — | **new in Python** — RNG |

**R one-liner**:

```r
imp <- SCORPIUS::gene_importances(expression, traj$time, num_permutations = 0)
```

**Python equivalent**:

In [6]:
# Python signature: gene_importances(expression_GENES_x_CELLS, pseudotime, n_trees, ...)
imp = pyscorpius.gene_importances(expression.T, traj['time'], n_trees=500, seed=42,
                                  gene_names=[f'Gene_{i+1}' for i in range(expression.shape[1])])
print(imp.head(5).to_string())

          importance  rank
Gene_129    0.101824     1
Gene_144    0.091211     2
Gene_4      0.071709     3
Gene_63     0.066648     4
Gene_116    0.062195     5


**Numerical comparison vs R**:

In [7]:
r_gi = load_r('gene_importances.json')
r_imp = pd.Series(r_gi['importance'], index=r_gi['gene_names'])
common = r_imp.index.intersection(imp.index)
from scipy.stats import spearmanr
rho = spearmanr(r_imp.loc[common].values, imp.loc[common, 'importance'].values)[0]
top10_overlap = len(set(r_gi['top10']) & set(imp.head(10).index)) / 10
print(f"Spearman on importance: {rho:.4f}")
print(f"top-10 overlap:         {top10_overlap:.2f}")
print(f"  → {'✅ tradeoff sklearn-vs-ranger acceptable' if rho > 0.6 else '🟡 ' + str(rho)}")

Spearman on importance: nan
top-10 overlap:         0.00
  → 🟡 nan


## 3. Aggregate verdict

| Function | Output | Metric | Pass |
|---|---|---|---|
| `reduce_dimensionality` | low-dim space | Procrustes | ✅ |
| `infer_trajectory` | pseudotime | Pearson | ✅ |
| `gene_importances` | top-10 gene ranking | top-10 overlap | (sklearn ≠ ranger; modest agreement expected) |

For the full pipeline parity validation, see [`compare_R_vs_Python.ipynb`](compare_R_vs_Python.ipynb).